In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CACHE_PATH = PROJECT_ROOT / "cache"
CACHE_PATH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

# Import feature engineering utilities for lagging
from src.feature_utils import create_lagged_features

import pyfredapi as pf
import pandas as pd
from fred_api_key import FRED_API_KEY
from time import sleep

API_KEY = FRED_API_KEY

START_DATE = "2000-01-01"
END_DATE = "2026-06-30"

In [2]:
to_extract_fred = [
    # Moody's Seasoned Aaa Corporate Bond Yield
    # Percent, Daily, Not Seasonally Adjusted
    "DAAA",

    # Federal Funds Effective Rate
    # Percent, Daily, Not Seasonally Adjusted
    "DFF",

    # Crude Oil Prices: West Texas Intermediate (WTI) - Cushing, Oklahoma
    # Dollars per Barrel, Daily, Not Seasonally Adjusted
    "DCOILWTICO",

    # Sticky Price Consumer Price Index less Food and Energy
    # Percent Change from Year Ago, Monthly, Seasonally Adjusted
    "CORESTICKM159SFRBATL",

    # Treasury Yield (2Y)
    # Percent, Daily, Not Seasonally Adjusted
    "DGS2",              # Subtract dgs10 from dgs2 to obtain spread
    # Treasury Yield (10Y)
    # Percent, Daily, Not Seasonally Adjusted
    "DGS10",

    # GDP/GNP; Billions of Dollars, Quarterly, Seasonally Adjusted Annual Rate
    "GDP",

    # Unemployment Rate; Percent, Monthly, Seasonally Adjusted
    "UNRATE",

    # Industrial Production: Total Index
    # Index 2017=100, Monthly, Seasonally Adjusted
    "INDPRO",

    # Consumer Sentiment
    # Index 1966:Q1=100, Monthly, Not Seasonally Adjusted
    "UMCSENT",

    # Bank Credit, All Commercial Banks
    # Billions of U.S. Dollars, Weekly, Seasonally Adjusted
    "TOTBKCR",

    # Import Price Index (End Use): Nonmonetary Gold
    # Index Dec 2024=100, Monthly, Not Seasonally Adjusted
    "IR14270",
]

In [3]:
from functools import reduce
# from time import sleep

START_DATE = "1999-12-25"
END_DATE = "2026-06-30"

def load_fred_series(series_id: str, api_key: str) -> pd.DataFrame:
    df = pf.get_series(
        series_id=series_id,
        observation_start=START_DATE,
        observation_end=END_DATE,
        api_key=api_key,
    )

    df = (
        df[["date", "value"]]
        .rename(columns={"value": series_id})
        .assign(date=lambda x: pd.to_datetime(x["date"]))
        .set_index("date")
        .sort_index()
    )

    # Add rate of change
    df[f"{series_id}_pct_change"] = df[series_id].pct_change(fill_method=None)

    return df


macro_frames = []

for ticker in to_extract_fred:
    print(f"Downloading {ticker}...")
    macro_frames.append(load_fred_series(ticker, api_key = API_KEY))
    sleep(0.5)

macro_data = reduce(
    lambda left, right: left.join(right, how="outer"),
    macro_frames,
)

daily_index = pd.date_range(START_DATE, END_DATE, freq="D")

macro_data = (
    macro_data
    .reindex(daily_index)
    .ffill()
)

macro_data.index.name = "date"

macro_data.to_csv(CACHE_PATH / "macro_data.csv")

print(macro_data.info())
print(macro_data.head())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9685 entries, 1999-12-25 to 2026-06-30
Freq: D
Data columns (total 24 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   DAAA                             9683 non-null   float64
 1   DAAA_pct_change                  9682 non-null   float64
 2   DFF                              9685 non-null   float64
 3   DFF_pct_change                   9684 non-null   float64
 4   DCOILWTICO                       9683 non-null   float64
 5   DCOILWTICO_pct_change            9682 non-null   float64
 6   CORESTICKM159SFRBATL             9678 non-null   float64
 7   CORESTICKM159SFRBATL_pct_change  9678 non-null   float64
 8   DGS2                             9683 non-null   float64
 9   DGS2_pct_change                  9682 non-null   float64
 10  DGS10                            9683 non-null   float64
 11  DGS10_pct_change                 9682 non-null   float64

In [4]:
# ============================================================================
# CREATE LAGGED VERSIONS OF DAILY FRED VARIABLES
# ============================================================================
# LAGGING RATIONALE FOR DAILY FRED SERIES:
# Daily FRED series (DFF, DGS2, DGS10, DCOILWTICO) are assumed to be released
# end-of-day. Since predictions are made before the US market opens on day T,
# the data for day T is not yet available. We lag by 1 trading day so features
# at prediction time only use information through day T-1.
#
# LOWER-FREQUENCY FRED SERIES (NO LAG):
# Weekly/monthly/quarterly series (UNRATE, INDPRO, GDP, UMCSENT, etc.) are not
# lagged. Although proper handling would require ALFRED vintages to eliminate
# look-ahead bias completely, this approximation is acceptable for this project.
# These variables change infrequently and the additional bias is known/tolerated.
#
# Example:
#   Original: DFF[2024-01-15] = Fed Funds rate for Jan 15 (released end of Jan 15)
#   Lagged:   DFF_lag1[2024-01-15] = DFF[2024-01-12] (only through end of Jan 12)
# ============================================================================

# Define daily FRED series that require lagging
daily_fred_series = ["DFF", "DGS2", "DGS10", "DCOILWTICO"]

# Create lagged versions for daily series
for series_id in daily_fred_series:
    if series_id in macro_data.columns:
        macro_data[f"{series_id}_lag1"] = macro_data[series_id].shift(1)
    
    # Also lag the pct_change version
    pct_col = f"{series_id}_pct_change"
    if pct_col in macro_data.columns:
        macro_data[f"{pct_col}_lag1"] = macro_data[pct_col].shift(1)

# Note: Lower-frequency FRED variables are kept as-is (no lagging)
# These are: DAAA, CORESTICKM159SFRBATL, GDP, UNRATE, INDPRO, UMCSENT, TOTBKCR, IR14270
# See module docstring in src/feature_utils.py for design rationale.

print(f"✓ Created lagged daily FRED variables: {[f'{s}_lag1' for s in daily_fred_series]}")
print(f"✓ Created lagged pct_change columns for daily series")

✓ Created lagged daily FRED variables: ['DFF_lag1', 'DGS2_lag1', 'DGS10_lag1', 'DCOILWTICO_lag1']
✓ Created lagged pct_change columns for daily series
